# SKAB EDA dan Persiapan Data, PDAM Pump Sentinel

Notebook ini mendemonstrasikan **Exploratory Data Analysis (EDA)** dan **Persiapan Data SKAB** untuk vertical slice akademik PDAM Pump Sentinel.
SKAB, Skoltech Anomaly Benchmark, dipakai sebagai *surrogate* public water circulation testbed agar pipeline dapat diuji tanpa membuka data operasional.
**Penting:** SKAB **bukan data operasional PDAM nyata**. Hasil pada fixture `tests/fixtures/skab_tiny.csv` hanya dipakai untuk kontrak dan demo, bukan bukti performa benchmark.

Alur yang ditunjukkan: missing policy, timestamp quality, korelasi dan heatmap korelasi, distribusi, rolling statistics, label ranges, label overlay, split manifest metadata/base_dir, event/range metrics, held-out test metrics, pemisahan changepoint, dan provenance.


## Setup Project Root

Agar notebook dapat dijalankan dari direktori mana pun, project root dicari dari `pyproject.toml` dan fixture `tests/fixtures/skab_tiny.csv`.
`PROJECT_ROOT` lalu ditambahkan ke `sys.path` agar import modul internal `ml.*` konsisten di notebook dan CI.


In [ ]:
import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        fixture = candidate / 'tests' / 'fixtures' / 'skab_tiny.csv'
        if (candidate / 'pyproject.toml').exists() and fixture.exists():
            return candidate
    raise FileNotFoundError('Project root with tests/fixtures/skab_tiny.csv was not found')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INPUT_CSV = PROJECT_ROOT / 'tests' / 'fixtures' / 'skab_tiny.csv'
ARTIFACT_DIR = Path('/tmp/pdam-pump-sentinel-skab-eda-data-prep')

print('Project root :', PROJECT_ROOT)
print('Input CSV    :', INPUT_CSV.resolve())
print('Artifact dir :', ARTIFACT_DIR.resolve())


## Memuat Fixture, Schema, dan Kebijakan Missing

`load_skab_csv` mem-parse `datetime` ke UTC, menghapus duplikasi timestamp dengan `keep='first'`, dan menolak timestamp yang tidak naik monoton.
Label `anomaly` dan `changepoint` dilengkapi dengan 0 bila tidak ada, tetapi fitur model tetap hanya memakai kolom sensor.
Kebijakan missing untuk sensor bersifat ketat, `allow_missing=False`, sehingga training tidak lanjut bila ada sensor kosong.


In [ ]:
from ml.datasets.skab_loader import SENSOR_COLUMNS, load_skab_csv, validate_sensor_values

df = load_skab_csv(INPUT_CSV)
missing_policy = validate_sensor_values(df, SENSOR_COLUMNS, allow_missing=False)

print('Jumlah baris       :', len(df))
print('Kolom              :', list(df.columns))
print('Sensor cols        :', SENSOR_COLUMNS)
print('Label cols         :', ['anomaly', 'changepoint'])
print('Missing policy     : allow_missing=False untuk sensor training')
print('Missing counts     :', missing_policy['missing_counts'])
print('Missing rates      :', missing_policy['missing_rates'])
print()
print(df.head())


## Generasi Laporan EDA Research-grade

`generate_skab_eda_report` sekarang menulis `summary.json`, `report.md`, `sensor_statistics.csv`, `missingness.csv`, `timestamp_quality.json`, `correlation_matrix.csv`, `correlation_heatmap.png`, `sensor_distributions.csv`, `rolling_statistics.csv`, `label_ranges.csv`, dan `label_overlay.csv`.
Artefak ini memberi jejak audit sebelum training, meliputi kualitas timestamp, missing policy aktual, distribusi sensor, korelasi antar sensor, heatmap korelasi, statistik rolling, range label, dan overlay label terhadap sensor.


In [ ]:
from ml.datasets.skab_eda import generate_skab_eda_report

EDA_ARTIFACT_FILES = [
    'summary.json',
    'report.md',
    'sensor_statistics.csv',
    'missingness.csv',
    'timestamp_quality.json',
    'correlation_matrix.csv',
    'correlation_heatmap.png',
    'sensor_distributions.csv',
    'rolling_statistics.csv',
    'label_ranges.csv',
    'label_overlay.csv',
    'label_overlay_plot.json',
    'label_overlay_plot.html',
]

artifacts = generate_skab_eda_report(
    input_path=INPUT_CSV,
    split_manifest_path=None,
    output_dir=ARTIFACT_DIR,
    include_plots=True,
)

print('Artefak EDA yang diharapkan:', EDA_ARTIFACT_FILES)
print('Artefak yang dihasilkan:')
for name, path in artifacts.items():
    print(f'  {name}: {path}')


## Inspeksi Ringkasan, Timestamp Quality, dan Missingness

Bagian ini membaca `summary.json`, `report.md`, `timestamp_quality.json`, dan `missingness.csv`.
Tujuannya bukan menilai performa model, melainkan memastikan kontrak data, timestamp, dan kebijakan missing terlihat eksplisit sebelum windowing dan training.


In [ ]:
%matplotlib inline

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

summary = json.loads(artifacts['input_summary_json'].read_text(encoding='utf-8'))
timestamp_quality = json.loads(artifacts['input_timestamp_quality_json'].read_text(encoding='utf-8'))
missingness = pd.read_csv(artifacts['input_missingness_csv'])
report_text = artifacts['input_report_md'].read_text(encoding='utf-8')

print('Ringkasan EDA (summary.json):')
print(json.dumps({
    'row_count': summary['row_count'],
    'anomaly_count': summary['anomaly_count'],
    'changepoint_count': summary['changepoint_count'],
    'time_range': summary['time_range'],
    'constant_column_flags': summary['constant_column_flags'],
}, indent=2, ensure_ascii=False))
print()
print('Timestamp quality (timestamp_quality.json):')
print(json.dumps(timestamp_quality, indent=2, ensure_ascii=False))
print()
print('Missingness (missingness.csv):')
print(missingness)
print()
print('Report preview (report.md):')
print('\n'.join(report_text.splitlines()[:16]))


## Inspeksi Korelasi, Distribusi, Rolling Stats, dan Label Overlay

`correlation_matrix.csv` membantu membaca hubungan antar sensor.
`sensor_distributions.csv` menyimpan kuantil p01 sampai p99 dan IQR untuk tiap sensor.
`rolling_statistics.csv` memberi ringkasan rolling mean dan rolling std.
`label_ranges.csv` dan `label_overlay.csv` memperlihatkan range label anomaly/changepoint dan overlay baris, sehingga changepoint dapat dipisahkan dari target anomaly.


In [ ]:
correlation_matrix = pd.read_csv(artifacts['input_correlation_matrix_csv'])
sensor_distributions = pd.read_csv(artifacts['input_sensor_distributions_csv'])
rolling_statistics = pd.read_csv(artifacts['input_rolling_statistics_csv'])
label_ranges = pd.read_csv(artifacts['input_label_ranges_csv'])
label_overlay = pd.read_csv(artifacts['input_label_overlay_csv'])

print('Correlation matrix (correlation_matrix.csv):')
print(correlation_matrix)
print()
print('Sensor distributions (sensor_distributions.csv):')
print(sensor_distributions[['sensor', 'missing_rate', 'p50', 'p95', 'iqr']])
print()
print('Rolling statistics (rolling_statistics.csv):')
print(rolling_statistics[['sensor', 'window', 'rolling_mean_last', 'rolling_std_last']])
print()
print('Label ranges (label_ranges.csv):')
print(label_ranges)
print()
print('Label overlay (label_overlay.csv):')
print(label_overlay[['row_index', 'timestamp', 'anomaly', 'changepoint', 'label']])


## Correlation Heatmap dengan `seaborn` / `sns`

Heatmap berikut memakai pola EDA konvensional `sns.heatmap(...)` untuk memvisualisasikan `correlation_matrix.csv`.
Grafik dirender langsung di output sel notebook. Berkas kanonik `correlation_heatmap.png` tetap dihasilkan oleh `generate_skab_eda_report(include_plots=True)` di pipeline, bukan dari sel ini.


In [ ]:
heatmap_values = correlation_matrix.set_index('sensor').astype(float)
mask = np.triu(np.ones_like(heatmap_values, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    heatmap_values,
    mask=mask,
    annot=True,
    cmap='coolwarm',
    fmt='.2f',
    square=True,
    vmin=-1.0,
    vmax=1.0,
    linewidths=0.5,
    cbar_kws={'label': 'Pearson r'},
    ax=ax,
)
ax.set_title('Correlation Heatmap antar Sensor SKAB')
ax.set_xlabel('Sensor')
ax.set_ylabel('Sensor')
ax.tick_params(axis='x', rotation=45)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
fig.tight_layout()
plt.show()


## Distribusi Sensor (Histogram)

Histogram per sensor memberi gambaran awal sebaran nilai. Pada fixture tiga baris,
grafik ini hanya ilustratif dan bukan karakteristik distribusi operasional PDAM.
Grafik dirender langsung di output sel notebook (tanpa menyimpan berkas gambar).


In [ ]:
sensor_cols = list(SENSOR_COLUMNS)
n_cols = 4
n_rows = (len(sensor_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()
for ax, col in zip(axes, sensor_cols):
    sns.histplot(df[col], bins=5, kde=False, color='#3498db', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')
for ax in axes[len(sensor_cols):]:
    ax.set_visible(False)
fig.suptitle('Distribusi Nilai Sensor, skab_tiny.csv')
fig.tight_layout()
plt.show()


## Boxplot Sensor Terstandardisasi

Karena skala antar sensor sangat berbeda (mis. Voltage vs Accelerometer), nilai
dibakukan ke z-score sebelum Boxplot agar sebaran dan outlier dapat dibandingkan.
Grafik dirender langsung di output sel notebook (tanpa menyimpan berkas gambar).


In [ ]:
sensor_frame = df[list(SENSOR_COLUMNS)].astype(float)
sensor_std = sensor_frame.std(ddof=0).replace(0.0, 1.0)
sensor_z = (sensor_frame - sensor_frame.mean()) / sensor_std
sensor_z_long = sensor_z.melt(var_name='sensor', value_name='z_score')

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=sensor_z_long, x='sensor', y='z_score', hue='sensor', legend=False, palette='Set2', ax=ax)
ax.set_title('Boxplot Sensor (z-score), skala dibakukan')
ax.set_xlabel('Sensor')
ax.set_ylabel('z-score')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.show()


## Bar Missingness per Kolom

Visualisasi `missing_rate` tiap kolom dari `missingness.csv`. Kebijakan sensor
training bersifat ketat (`allow_missing=False`), sehingga bar yang tidak nol
akan menghentikan training.
Grafik dirender langsung di output sel notebook (tanpa menyimpan berkas gambar).


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=missingness, x='column', y='missing_rate', hue='column', legend=False, palette='viridis', ax=ax)
ax.set_title('Missing Rate per Kolom')
ax.set_xlabel('Kolom')
ax.set_ylabel('Missing rate')
ax.set_ylim(0, 1)
if float(missingness['missing_rate'].max()) == 0.0:
    ax.text(0.5, 0.5, 'Tidak ada missing value\n(semua kolom lengkap)', transform=ax.transAxes,
            ha='center', va='center', fontsize=14, color='#2ecc71', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.show()


## Rolling Mean dan Rolling Std per Sensor

Bar berikut membandingkan `rolling_mean_last` dan `rolling_std_last` per sensor
dari `rolling_statistics.csv`.
Grafik dirender langsung di output sel notebook (tanpa menyimpan berkas gambar).


In [ ]:
rolling_long = rolling_statistics.melt(
    id_vars='sensor',
    value_vars=['rolling_mean_last', 'rolling_std_last'],
    var_name='statistic',
    value_name='value',
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=rolling_long, x='sensor', y='value', hue='statistic', palette='mako', ax=ax)
ax.set_title('Rolling Mean/Std Terakhir per Sensor')
ax.set_xlabel('Sensor')
ax.set_ylabel('Nilai (skala log)')
ax.set_yscale('log')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
plt.show()


## Time Series Sensor (Small Multiples)

Grafik garis tiap sensor terhadap waktu. Untuk fixture tiga baris ini hanya
ilustrasi alur eksekusi, bukan pola operasional.
Grafik dirender langsung di output sel notebook (tanpa menyimpan berkas gambar).


In [ ]:
ts_index = df['datetime'] if 'datetime' in df.columns else range(len(df))
sensor_cols = list(SENSOR_COLUMNS)
n_cols = 4
n_rows = (len(sensor_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows), sharex=True)
axes = axes.flatten()
for ax, col in zip(axes, sensor_cols):
    ax.plot(ts_index, df[col], marker='o', color='#2ecc71')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)
for ax in axes[len(sensor_cols):]:
    ax.set_visible(False)
fig.suptitle('Time Series Sensor, skab_tiny.csv')
fig.tight_layout()
plt.show()


## Time Series dengan Label Overlay, Anomaly dan Changepoint

Grafik nyata pengganti preview SVG statis sebelumnya. Dua sensor (Current dan
Temperature) dibakukan ke z-score, lalu baris `anomaly` ditandai titik merah dan
baris `changepoint` ditandai garis putus-putus. Tetap hanya ilustrasi fixture,
bukan karakteristik operasional PDAM.
Grafik dirender langsung di output sel notebook (tanpa menyimpan berkas gambar).


In [ ]:
overlay_sensors = ['Current', 'Temperature']
sensor_frame = df[list(SENSOR_COLUMNS)].astype(float)
sensor_std = sensor_frame.std(ddof=0).replace(0.0, 1.0)
sensor_z = (sensor_frame - sensor_frame.mean()) / sensor_std

x = label_overlay['row_index']
anomaly_rows = label_overlay.loc[label_overlay['anomaly'] == 1, 'row_index']
changepoint_rows = label_overlay.loc[label_overlay['changepoint'] == 1, 'row_index']

fig, ax = plt.subplots(figsize=(12, 6))
for col, color in zip(overlay_sensors, ['#2ecc71', '#e67e22']):
    ax.plot(x, sensor_z[col], marker='o', label=col, color=color)
for position, row in enumerate(anomaly_rows):
    ax.scatter(row, sensor_z.loc[row, overlay_sensors[0]], color='#d62728', s=120, zorder=5, label='anomaly' if position == 0 else None)
for position, row in enumerate(changepoint_rows):
    ax.axvline(row, color='#ff7f0e', linestyle='--', label='changepoint' if position == 0 else None)
ax.set_title('Time Series (z-score) dengan Overlay Anomaly/Changepoint')
ax.set_xlabel('row_index')
ax.set_ylabel('z-score')
ax.legend(loc='best')
fig.tight_layout()
plt.show()


## Membuat Split Manifest dengan Metadata dan `base_dir`

Untuk simulasi train/validation/test split, fixture disalin ke tiga nama berkas berbeda agar tidak ada duplikasi antar split.
Manifest memakai metadata `schema_version`, `dataset_kind`, `name`, `notes`, dan `base_dir`.
`base_dir` membuat path file tetap relatif terhadap direktori manifest, sehingga manifest portabel dan mudah diaudit.


In [ ]:
import shutil

TEMP_DIR = ARTIFACT_DIR
MANIFEST_DIR = TEMP_DIR / 'split_manifest'
MANIFEST_DATA_DIR = MANIFEST_DIR / 'files'
MANIFEST_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_copy = MANIFEST_DATA_DIR / 'skab_tiny_train.csv'
val_copy = MANIFEST_DATA_DIR / 'skab_tiny_validation.csv'
test_copy = MANIFEST_DATA_DIR / 'skab_tiny_test.csv'
shutil.copy(INPUT_CSV, train_copy)
shutil.copy(INPUT_CSV, val_copy)
shutil.copy(INPUT_CSV, test_copy)

manifest_content = {
    'schema_version': 1,
    'dataset_kind': 'skab-surrogate-demo',
    'name': 'skab_tiny_demo_manifest',
    'base_dir': 'files',
    'notes': {
        'purpose': 'academic demo only',
        'caveat': 'SKAB fixture is not real PDAM operational data',
    },
    'train': ['skab_tiny_train.csv'],
    'validation': ['skab_tiny_validation.csv'],
    'test': ['skab_tiny_test.csv'],
}

MANIFEST_PATH = MANIFEST_DIR / 'manifest.json'
MANIFEST_PATH.write_text(json.dumps(manifest_content, indent=2), encoding='utf-8')

print('Manifest path:', MANIFEST_PATH.resolve())
print('Manifest base_dir:', manifest_content['base_dir'])
print('Train copy:', train_copy.resolve())
print('Validation copy:', val_copy.resolve())
print('Test copy:', test_copy.resolve())


## Memuat Split Manifest dan EDA per Split

`load_skab_split_manifest` memvalidasi metadata, memastikan `base_dir` tetap relatif, memastikan semua entry berada di bawah `base_dir`, dan menolak file yang sama muncul di lebih dari satu split.
EDA manifest menghasilkan artefak per split, misalnya `train_summary_json`, `validation_label_ranges_csv`, dan `test_timestamp_quality_json`.


In [ ]:
from ml.datasets.skab_manifest import load_skab_split_manifest

manifest = load_skab_split_manifest(MANIFEST_PATH)

print('Manifest metadata:')
print('  schema_version :', manifest.schema_version)
print('  dataset_kind   :', manifest.dataset_kind)
print('  name           :', manifest.name)
print('  base_dir       :', manifest.base_dir)
print('  resolved base  :', manifest._base_dir)
print('  notes          :', manifest.notes)
print('Resolved payload:', manifest.to_payload())

split_eda_artifacts = generate_skab_eda_report(
    input_path=None,
    split_manifest_path=MANIFEST_PATH,
    output_dir=ARTIFACT_DIR / 'split_eda',
    include_plots=False,
)

print('Split EDA artifact keys:')
print(sorted(split_eda_artifacts.keys()))
print()
print('Validation label ranges:')
print(pd.read_csv(split_eda_artifacts['validation_label_ranges_csv']))
print()
print('Test timestamp quality:')
print(json.loads(split_eda_artifacts['test_timestamp_quality_json'].read_text(encoding='utf-8')))


## Demonstrasi Windowing dan Pemisahan Label

`build_sensor_windows` membangun fitur hanya dari `SENSOR_COLUMNS`.
Label `anomaly` menjadi target window, sedangkan `changepoint` disimpan sebagai mask transient terpisah.
Pemisahan changepoint ini penting karena changepoint bukan fitur dan bukan target utama alert, melainkan konteks evaluasi yang bisa dikeluarkan dari metrik dengan suffix `_excluding_transient`.


In [ ]:
from ml.features.windowing import build_sensor_windows

windowed = build_sensor_windows(
    frame=df,
    window_size=1,
    stride=1,
    sensor_columns=SENSOR_COLUMNS,
)

print('Features shape     :', windowed.features.shape)
print('Anomaly labels     :', windowed.labels.tolist())
print('Changepoint mask   :', windowed.changepoints.tolist())
print('Timestamps         :', windowed.timestamps.tolist())
print('Sensor columns     :', windowed.sensor_columns)
print('Window size        :', windowed.window_size)
print('Stride             :', windowed.stride)


## Held-out Test Metrics, Event Metrics, dan Provenance

Training PCA split-manifest memakai train normal untuk fitting, validation normal untuk kalibrasi threshold, dan test sebagai held-out split.
`metrics.json` berisi metrik validation di top level, metrik test dengan prefix `test_`, metrik event seperti `event_count`, `event_recall`, `missed_events`, `false_alarm_events`, dan `mean_detection_delay_windows`, serta versi `_excluding_transient` untuk pemisahan changepoint.
`metadata.json` menyimpan `metric_protocol`, `split`, `test_split_held_out`, dan `provenance` agar konfigurasi, input files, hash, dan versi paket dapat diaudit.


In [ ]:
from ml.training.train_pca import PcaTrainingConfig, train_pca_from_skab

PCA_ARTIFACT_DIR = TEMP_DIR / 'pca_split_artifacts'
pca_result = train_pca_from_skab(
    PcaTrainingConfig(
        input_path=PCA_ARTIFACT_DIR,
        output_dir=PCA_ARTIFACT_DIR,
        split_manifest_path=MANIFEST_PATH,
        window_size=1,
        stride=1,
        threshold_quantile=0.95,
        log_mlflow=False,
    )
)

metrics = json.loads(pca_result.artifact_paths['metrics'].read_text(encoding='utf-8'))
metadata = json.loads(pca_result.artifact_paths['metadata'].read_text(encoding='utf-8'))
split_manifest_artifact = json.loads(pca_result.artifact_paths['split_manifest'].read_text(encoding='utf-8'))

metric_keys = [
    'precision', 'recall', 'f1', 'false_alarm_rate',
    'event_count', 'event_recall', 'missed_events', 'false_alarm_events', 'mean_detection_delay_windows',
    'precision_excluding_transient', 'event_recall_excluding_transient',
    'test_precision', 'test_recall', 'test_f1', 'test_false_alarm_rate',
    'test_event_count', 'test_event_recall', 'test_missed_events',
    'test_false_alarm_events', 'test_mean_detection_delay_windows',
    'test_precision_excluding_transient', 'test_event_recall_excluding_transient',
]

print('Metrics protocol keys:', sorted(metadata['metric_protocol'].keys()))
print('Held-out test split:', metadata['test_split_held_out'])
print('Split metadata:', metadata['split'])
print('Resolved split_manifest.json:', split_manifest_artifact)
print('Selected metrics:')
print(json.dumps({key: metrics.get(key) for key in metric_keys if key in metrics}, indent=2))
print('Provenance keys:', sorted(metadata['provenance'].keys()))
print('Provenance input files:', metadata['provenance'].get('input_files'))

if 'test_scores' in pca_result.artifact_paths:
    print('Held-out test scores preview:')
    print(pd.read_csv(pca_result.artifact_paths['test_scores']).head())


## Catatan Akademik

- SKAB adalah dataset publik dari water circulation testbed dan dipakai sebagai surrogate untuk verifikasi pipeline. Dataset ini **bukan data operasional PDAM nyata**.
- Fixture `skab_tiny.csv` hanya berisi tiga baris, sehingga semua metrik, grafik, dan threshold di notebook ini bersifat smoke test dan kontrak eksekusi.
- Jangan mengklaim hasil benchmark dari fixture kecil. Untuk studi yang layak dilaporkan, gunakan SKAB lengkap atau data telemetry PDAM yang sah, dengan split berbasis file dan test held-out.
- Artefak EDA dan training ditulis ke `/tmp/pdam-pump-sentinel-skab-eda-data-prep` agar working tree tetap bersih.
- Changelog metodologi yang perlu dibawa ke laporan: missing policy eksplisit, timestamp quality, korelasi/distribusi/rolling stats, label ranges/overlay, manifest metadata/base_dir, event metrics, held-out test metrics, changepoint separation, dan provenance.
